<a href="https://colab.research.google.com/github/DA15-Lab/team3-ping-analytics/blob/main/notebooks/polls_usercandidate_%EC%A0%84%EC%B2%98%EB%A6%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# # ===== 팀 프로젝트 시작 셀 =====
# import os, sys
# from google.colab import userdata

# TOKEN = userdata.get('GITHUB_TOKEN')
# NAME = "본인이름"
# REPO_URL = "github.com/DA15-Lab/team3-ping-analytics.git"
# PROJECT_DIR = "/content/team3-ping-analytics"

# if not os.path.exists(PROJECT_DIR):
#     !git clone -q https://{TOKEN}@{REPO_URL}
#     %cd {PROJECT_DIR}
# else:
#     %cd {PROJECT_DIR}
#     !git pull -q

# !git config user.email "{NAME}@example.com"
# !git config user.name "{NAME}"

# !pip install -q -r requirements.txt

# sys.path.append("/content/team3-ping-analytics/src")
# import preprocess

# print(preprocess.module_test())
# print("✅ 준비 완료")

/content/team3-ping-analytics
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 96.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
✅ 모듈 설치 완료
✅ 준비 완료


In [11]:
# ===== 작업 저장 셀 =====
%cd {PROJECT_DIR}

from google.colab import _message
import json

with open('notebooks/yurim_analysis.ipynb', 'w') as f:
    json.dump(_message.blocking_request('get_ipynb', timeout_sec=30)['ipynb'], f)

!pip freeze | grep -v "google-colab" > requirements.txt

!git add .
!git commit -m "작업한 내용 메시지"
!git config pull.rebase false
!git pull
!git push

/content/team3-ping-analytics
[main 85d97c8] 작업한 내용 메시지
Already up to date.
Enumerating objects: 18, done.
Counting objects: 100% (17/17), done.
Delta compression using up to 2 threads
Compressing objects: 100% (12/12), done.
Writing objects: 100% (12/12), 4.96 KiB | 1.24 MiB/s, done.
Total 12 (delta 8), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (8/8), completed with 3 local objects.
To https://github.com/DA15-Lab/team3-ping-analytics.git
   2ff5294..85d97c8  main -> main


# 대주제2-소주제3 전처리 — polls_usercandidate

(관계 친밀도가 신규 유저 활성화·이탈에 미치는 영향 / 네트워크의 질)

In [ ]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

Mounted at /content/drive


## 전처리 — polls_usercandidate

질문 1건(question_piece_id)당 평균 3.92명의 후보가 제시되는 테이블. accounts_userquestionrecord와 조인해 이 후보가 어떤 viewer에게 제시됐는지 파악하고, viewer와 후보의 group_id(반)를 비교해 매칭 구조(같은 반/타 반/타 학교)를 판정합니다.

In [ ]:
# 원본 데이터 경로 — 본인 Drive 폴더 구조에 맞게 수정
DATA_DIR = '/content/drive/MyDrive/고급 프로젝트/csv'   # TODO: 실제 경로로 교체

candidate = pd.read_csv(os.path.join(DATA_DIR, 'polls_usercandidate.csv'))
uqr = pd.read_csv(os.path.join(DATA_DIR, 'accounts_userquestionrecord.csv'))
user = pd.read_csv(os.path.join(DATA_DIR, 'accounts_user.csv'))

print(candidate.shape, uqr.shape, user.shape)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/고급 프로젝트/csv/polls_usercandidate.csv'

In [ ]:
# 날짜 컬럼 변환
candidate['created_at'] = pd.to_datetime(candidate['created_at'])

# 테스트/관리자 계정 제외 (전체 677,085건 중 4건)
user = user[(user['is_superuser'] == 0) & (user['is_staff'] == 0)].copy()

# question_piece_id -> 실제로 이 질문을 받은 유저(viewer) 매핑
viewer_map = uqr.set_index('question_piece_id')['user_id']
candidate['viewer_id'] = candidate['question_piece_id'].map(viewer_map)

# 후보(user_id)와 viewer 각각의 group_id(반) 붙이기
candidate = candidate.merge(
    user[['id', 'group_id']].rename(columns={'id': 'user_id', 'group_id': 'candidate_group_id'}),
    on='user_id', how='left'
)
candidate = candidate.merge(
    user[['id', 'group_id']].rename(columns={'id': 'viewer_id', 'group_id': 'viewer_group_id'}),
    on='viewer_id', how='left'
)

candidate['same_class'] = candidate['candidate_group_id'] == candidate['viewer_group_id']

candidate[['question_piece_id', 'viewer_id', 'user_id', 'candidate_group_id', 'viewer_group_id', 'same_class']].head()

In [ ]:
# 매칭 구조 요약 (가이드 문서 수치: 후보의 70%가 타 반/타 학교)
print(candidate['same_class'].value_counts(normalize=True))